# pySCENIC

In [13]:
print("Starting...")

import sys
print(sys.executable)

Starting...
/opt/anaconda3/envs/3.10/bin/python


In [24]:
# # Py 3.10

# # One-time install (uncomment to run, then RESTART the kernel)
# import os
# os.environ["SETUPTOOLS_USE_DISTUTILS"] = "stdlib"
# %pip install "numpy==1.26.4" "pandas==2.1.4"
# %pip install "dask==2023.12.1" "distributed==2023.12.1"
# %pip install --no-cache-dir pyscenic==0.12.1 loompy
# !conda install -n 3.10 --force-reinstall pip "setuptools<70" -c conda-forge -y

In [19]:
# import re, pathlib, pyscenic
# pkg = pathlib.Path(pyscenic.__file__).parent
# changed = []
# for f in pkg.rglob("*.py"):
#     s = f.read_text(); o = s
#     s = re.sub(r"np\.(object|bool|int|float|str)\b", r"\1", s)   # NumPy 1.24
#     s = s.replace(".iteritems()", ".items()")                    # pandas 2.0
#     s = re.sub(r"np\.msort\(", "np.sort(", s)                    # NumPy 2.0
#     if s != o:
#         f.write_text(s); changed.append(f.name)
# print("Patched:", changed or "nothing (already clean)")

import re
import pathlib
import sys

# Get the site-packages directory for the active 3.10 environment
site_packages = pathlib.Path(sys.executable).parent.parent / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"

print(f"Patching site-packages at: {site_packages}")

changed = []

# Iterate over all .py files in site-packages (including pySCENIC, arboreto, loompy, etc.)
for f in site_packages.rglob("*.py"):
    try:
        s = f.read_text(encoding="utf-8")
        o = s
        
        # 1. NumPy 1.24 deprecated alias removals (np.bool, np.int, np.float, etc.)
        s = re.sub(r"\bnp\.(object|bool|int|float|str)\b", r"\1", s)
        
        # 2. Pandas 2.0 removal of .iteritems()
        s = s.replace(".iteritems()", ".items()")
        
        # 3. NumPy 2.0 removal of np.msort
        s = re.sub(r"\bnp\.msort\(", "np.sort(", s)
        
        if s != o:
            f.write_text(s, encoding="utf-8")
            changed.append(str(f.relative_to(site_packages)))
    except (UnicodeDecodeError, PermissionError):
        # Skip non-UTF8 or read-only files if any
        continue

print(f"Patched {len(changed)} file(s):")
for path in changed:
    print(f"  - {path}")

Patching site-packages at: /opt/anaconda3/envs/3.10/lib/python3.10/site-packages
Patched 19 file(s):
  - boltons/cacheutils.py
  - boltons/dictutils.py
  - boltons/urlutils.py
  - dask/dataframe/core.py
  - dask/dataframe/tests/test_dataframe.py
  - dask/array/tests/test_reductions.py
  - debugpy/_vendored/pydevd/pydevd_attach_to_process/winappdbg/registry.py
  - numpy_groupies/utils.py
  - numpy/__init__.py
  - numpy/lib/function_base.py
  - requests/cookies.py
  - pip/_vendor/requests/cookies.py
  - pip/_vendor/urllib3/_collections.py
  - sklearn/feature_extraction/_hash.py
  - urllib3/_collections.py
  - scipy/linalg/_basic.py
  - scipy/special/_basic.py
  - dill/tests/test_source.py
  - pandas/tests/io/excel/test_writers.py


In [4]:
import pyscenic
import json
import zlib
import base64
import loompy as lp
import pandas as pd

from dask.distributed import Client, LocalCluster
from arboreto.algo import grnboost2
from arboreto.utils import load_tf_names

from pyscenic.binarization import binarize

# Version sanity check
import dask, numpy
print("pyscenic", pyscenic.__version__, "| dask", dask.__version__,
      "| numpy", numpy.__version__, "| pandas", pd.__version__)

pyscenic 0.12.1 | dask 2023.12.1 | numpy 1.26.4 | pandas 2.1.4


In [5]:
print("Extracting the expression matrix from the Loom file...")
with lp.connect("Seurat.loom", mode="r", validate=False) as ds:
    pdf = pd.DataFrame(ds[:, :].T, index=ds.ca.CellID, columns=ds.ra.Gene)

pdf.columns = pdf.columns.astype(str)
print(f"Expression matrix: {pdf.shape[0]} cells x {pdf.shape[1]} genes")

print("Loading Transcription Factor (TF) names...")
tfs = load_tf_names("hs_hgnc_tfs.txt")

print("Initializing Dask LocalCluster (multi-process for real parallelism)...")
import multiprocessing
n = multiprocessing.cpu_count()
cluster = LocalCluster(
    n_workers=max(n - 1, 1),   # one process per core (leave 1 free)
    threads_per_worker=1,       # sklearn GBM is GIL-bound; threads don't help
    processes=True,             # processes = actual parallel CPU use
    memory_limit="4GB",
)
client = Client(cluster)
print(f"Cluster up: {len(cluster.workers)} workers | dashboard {client.dashboard_link}")

try:
    print("Launching GRNBoost2 (Inferring regulatory networks)...")
    network = grnboost2(
        expression_data=pdf,
        tf_names=tfs,
        client_or_address=client,
        verbose=True,
    )

    print("Saving the inferred network to CSV...")
    network.to_csv("adj.csv", index=False)
    print("Success! The results have been saved to 'adj.csv'")

finally:
    # Clean up and release system resources gracefully
    print("Shutting down the Dask cluster...")
    client.close()
    cluster.close()

Extracting the expression matrix from the Loom file...
Expression matrix: 22061 cells x 347 genes
Loading Transcription Factor (TF) names...
Initializing Dask LocalCluster (multi-process for real parallelism)...
Cluster up: 9 workers | dashboard http://127.0.0.1:8787/status
Launching GRNBoost2 (Inferring regulatory networks)...
preparing dask client
parsing input
creating dask graph
9 partitions
computing dask graph


/opt/anaconda3/envs/3.10/lib/python3.10/site-packages/distributed/client.py:3162: UserWarning: Sending large graph of size 49.49 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


not shutting down client, client was created externally
finished
Saving the inferred network to CSV...
Success! The results have been saved to 'adj.csv'
Shutting down the Dask cluster...


In [25]:
!/opt/anaconda3/envs/3.10/bin/pyscenic ctx adj.csv \
    *feather \
    --annotations_fname motifs-v9-nr.hgnc-m0.001-o0.0.tbl \
    --expression_mtx_fname Seurat.loom \
    --output reg.csv \
    --mask_dropouts \
    --num_workers 8


2026-08-26 13:21:11,419 - pyscenic.cli.pyscenic - INFO - Creating modules.

2026-08-26 13:21:11,423 - pyscenic.cli.pyscenic - INFO - Loading expression matrix.

2026-08-26 13:21:11,650 - pyscenic.utils - INFO - Calculating Pearson correlations.

2026-08-26 13:21:11,651 - pyscenic.utils - WARNING - Note on correlation calculation: the default behaviour for calculating the correlations has changed after pySCENIC verion 0.9.16. Previously, the default was to calculate the correlation between a TF and target gene using only cells with non-zero expression values (mask_dropouts=True). The current default is now to use all cells to match the behavior of the R verision of SCENIC. The original settings can be retained by setting 'rho_mask_dropouts=True' in the modules_from_adjacencies function, or '--mask_dropouts' from the CLI.
	Dropout masking is currently set to [True].

2026-08-26 13:21:11,721 - pyscenic.utils - INFO - Creating modules.

2026-08-26 13:21:11,942 - pyscenic.cli.pyscenic - IN

In [26]:
# AUCell
!/opt/anaconda3/envs/3.10/bin/pyscenic aucell Seurat.loom reg.csv --output Seurat_out.loom --num_workers 8


2026-08-26 13:24:26,540 - pyscenic.cli.pyscenic - INFO - Loading expression matrix.

2026-08-26 13:24:26,738 - pyscenic.cli.pyscenic - INFO - Loading gene signatures.
Create regulons from a dataframe of enriched features.
Additional columns saved: []

2026-08-26 13:24:26,860 - pyscenic.cli.pyscenic - INFO - Calculating cellular enrichment.

2026-08-26 13:24:56,886 - pyscenic.cli.pyscenic - INFO - Writing results to file.


In [27]:
# Binarize the AUCell output
print("Extracting AUC matrix from Loom...")

with lp.connect("Seurat_out.loom", mode="r", validate=False) as lf:
    auc_mtx = pd.DataFrame(lf.ca.RegulonsAUC, index=lf.ca.CellID)

auc_mtx.index.names = ["Cells"]

print("Saving raw AUC matrix...")
auc_mtx.to_csv("pySCENIC-AUC-Raw.csv")

print("Binarizing the AUC matrix...")
auc_mtx_binary, thresholds = binarize(auc_mtx, num_workers=8)

print("Saving binary AUC matrix...")
df_auc_mtx_binary = pd.DataFrame(auc_mtx_binary)
df_auc_mtx_binary.index.names = ["Cells"]

df_auc_mtx_binary.to_csv("pySCENIC-AUC-Binary.csv")
print("Success! Both 'pySCENIC-AUC-Raw.csv' and 'pySCENIC-AUC-Binary.csv' are ready.")

Extracting AUC matrix from Loom...
Saving raw AUC matrix...
Binarizing the AUC matrix...
Saving binary AUC matrix...
Success! Both 'pySCENIC-AUC-Raw.csv' and 'pySCENIC-AUC-Binary.csv' are ready.
